# 00 - Environment and unit tests
Records the exact Kaggle hardware/software identity and runs the unit tests. Kaggle numbers are **Kaggle** numbers, never Jetson numbers.

**Inputs:** none. **Outputs:** `/kaggle/working/artifacts/env/environment.json`, `status.json`, `unit_tests.txt`.

**Settings:** Internet ON, GPU on (T4/P100 is fine). Run all cells.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "pyyaml", "pillow"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
print(r.stdout[-500:])
# Do NOT upgrade torch/ultralytics here. Training notebooks pin ultralytics==8.3.40 explicitly.

In [ ]:
from pathlib import Path
from src.common.config import config_hash, load_yaml
from src.common.env_record import save_environment
from src.common.status import new_run_id, write_status

ART = Path("/kaggle/working/artifacts/env")
RUN_ID = new_run_id("env")
cfg_h = config_hash(load_yaml("configs/dataset.yaml"))
write_status(ART / "status.json", "00_environment", RUN_ID, "RUNNING", cfg_h)
env = save_environment(ART / "environment.json")
import json; print(json.dumps(env, indent=2))

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/unit"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
print(r.stdout[-3000:])
(ART / "unit_tests.txt").write_text(r.stdout)
state = "COMPLETE" if r.returncode == 0 else "FAILED"
write_status(ART / "status.json", "00_environment", RUN_ID, state, cfg_h, "unit tests " + ("passed" if r.returncode == 0 else "FAILED"))
print("STATE:", state)

## Next
Click **Save Version -> Save & Run All** if you want the outputs kept, then run notebook 01. Paste the printed environment and `STATE` back to the team/assistant.